# 01 - RTMPose CUDA validation

Notebook da US #10 / task #12 para validar o wrapper `RTMPoseEstimator` com CUDA e preparar a reprodutibilidade do PDJ reportado no paper do 3DSP.

Objetivos:
- instalar/checar dependências opcionais do RTMPose;
- confirmar que o backend CUDA está disponível no ONNX Runtime;
- rodar inferência em uma imagem real do 3DSP;
- rodar um smoke test em batch;
- calcular PDJ em uma amostra ou no dataset completo.

## Dependências opcionais

Execute esta célula uma vez no ambiente da máquina com GPU. Se o ambiente já tiver `rtmlib`, `onnxruntime-gpu` e as libs CUDA 12/cuDNN 9 acessíveis, pode pular.

A forma recomendada é instalar o extra oficial `onnxruntime-gpu[cuda,cudnn]`, que traz os pacotes NVIDIA necessários via pip. Se aparecer erro como `libcublasLt.so.12` ou `libcufft.so.11`, rode esta célula e reinicie o kernel.

In [6]:
# Instala RTMPose + ONNX Runtime GPU + runtime libs CUDA/cuDNN via extras oficiais.
# Depois de executar, reinicie o kernel do notebook.
!uv pip install --upgrade rtmlib "onnxruntime-gpu[cuda,cudnn]"

Using Python 3.12.9 environment at: /home/phaelzin/football-orient-pose/.venv
Resolved 17 packages in 133ms                                        
Prepared 1 package in 0.20ms                                             
Uninstalled 1 package in 10ms
Installed 1 package in 9ms                                  
 - numpy==2.4.4
 + numpy==2.4.6


## Setup do projeto

In [7]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_DIR = ROOT / "data" / "3dsp"
print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)

ROOT: /home/phaelzin/football-orient-pose
DATA_DIR: /home/phaelzin/football-orient-pose/data/3dsp


## Checagem do driver NVIDIA

Esta célula confirma se a máquina enxerga a GPU e qual versão de driver/CUDA é suportada pelo sistema.

In [8]:
!nvidia-smi

Tue May 19 14:19:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P8              1W /   60W |     392MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Checagem real de CUDA no ONNX Runtime

Para rodar em GPU, `CUDAExecutionProvider` precisa aparecer na lista **e** a biblioteca CUDA do ONNX Runtime precisa carregar sem erro.

O erro `libcublasLt.so.12` indica que o provider existe, mas as dependências CUDA 12/cuDNN 9 não estão carregáveis no ambiente atual.

In [9]:
import ctypes
from pathlib import Path

import onnxruntime as ort

print("onnxruntime:", ort.__version__)

# ONNX Runtime >= 1.21 consegue pré-carregar libs instaladas pelos pacotes nvidia-* do pip.
if hasattr(ort, "preload_dlls"):
    try:
        ort.preload_dlls()
        print("ort.preload_dlls(): ok")
    except Exception as exc:
        print("ort.preload_dlls(): falhou", repr(exc))

providers = ort.get_available_providers()
print("providers:", providers)
assert "CUDAExecutionProvider" in providers, (
    "CUDAExecutionProvider não está disponível. "
    "Instale onnxruntime-gpu e confira driver NVIDIA/CUDA/cuDNN."
)

cuda_provider_lib = Path(ort.__file__).parent / "capi" / "libonnxruntime_providers_cuda.so"
try:
    ctypes.CDLL(str(cuda_provider_lib))
    print("CUDA provider library: ok")
except OSError as exc:
    raise RuntimeError(
        "CUDAExecutionProvider aparece na lista, mas não conseguiu carregar as "
        "bibliotecas CUDA. Se o erro citar libcublasLt.so.12, libcufft.so.11 "
        "ou cuDNN, rode:\n"
        "  uv pip install --upgrade \"onnxruntime-gpu[cuda,cudnn]\"\n"
        "e reinicie o kernel. Se ainda falhar, verifique driver NVIDIA e CUDA 12."
    ) from exc


onnxruntime: 1.26.0
ort.preload_dlls(): ok
providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
CUDA provider library: ok


## Interpretação do fallback

Se a inferência funcionar mas aparecer warning `Failed to create CUDAExecutionProvider`, o resultado provavelmente foi calculado em CPU. Nesse caso, corrija a célula de CUDA acima antes de registrar tempos ou PDJ como execução em GPU.

## Inferência em uma imagem do 3DSP

In [10]:
import numpy as np

from football_orient_pose.estimators import RTMPoseEstimator
from football_orient_pose.utils.data_io import load_clip_image

image = load_clip_image(DATA_DIR / "train" / "00001", frame_idx=1)

estimator = RTMPoseEstimator(
    backend="onnxruntime",
    device="cuda",
)

kp_coco = estimator.predict(image)
kp_h3wb = estimator.predict_h3wb(image)

print("COCO shape:", kp_coco.shape)
print("H3WB shape:", kp_h3wb.shape)
print("confidence min/max:", float(kp_coco[:, 2].min()), float(kp_coco[:, 2].max()))

assert kp_coco.shape == (17, 3)
assert kp_h3wb.shape == (17, 2)
assert np.all(kp_coco[:, 2] > 0)

load /home/phaelzin/.cache/rtmlib/hub/checkpoints/rtmpose-x_simcc-body7_pt-body7_700e-384x288-71d7b7e9_20230629.onnx with onnxruntime backend
COCO shape: (17, 3)
H3WB shape: (17, 2)
confidence min/max: 0.32140588760375977 0.8793144226074219


2026-05-19 14:19:48.629182722 [W:onnxruntime:, session_state.cc:1367 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-05-19 14:19:48.629227522 [W:onnxruntime:, session_state.cc:1369 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


## Smoke test em batch

In [11]:
images = [load_clip_image(DATA_DIR / "train" / "00001", frame_idx=i) for i in range(1, 6)]
kp_batch = estimator.predict_batch(images)

print("Batch shape:", kp_batch.shape)
print("confidence min/max:", float(kp_batch[..., 2].min()), float(kp_batch[..., 2].max()))

assert kp_batch.shape == (5, 17, 3)
assert np.all(kp_batch[..., 2] > 0)

Batch shape: (5, 17, 3)
confidence min/max: 0.32140588760375977 0.9283201694488525


## Validação PDJ

Por padrão, a célula abaixo roda em poucos clips para smoke test. Para reproduzir a métrica do paper, mude `FULL_RUN = True` e execute em GPU.

Referência esperada para RTMPose no 3DSP: **PDJ@0.5 = 89.51%** e **AUC = 73.56%**.

In [12]:
from football_orient_pose.evaluation import compute_pdj, pdj_auc
from football_orient_pose.utils.data_io import iter_clips, load_keypoints_2d

FULL_RUN = False
MAX_CLIPS = None if FULL_RUN else 2

clip_dirs = iter_clips(DATA_DIR, split="train")
if MAX_CLIPS is not None:
    clip_dirs = clip_dirs[:MAX_CLIPS]

predictions = []
targets = []

for clip_dir in clip_dirs:
    for frame_idx in range(1, 21):
        image = load_clip_image(clip_dir, frame_idx)
        predictions.append(estimator.predict_h3wb(image))
        targets.append(load_keypoints_2d(clip_dir / "posture" / f"{frame_idx:03d}.json"))

predictions = np.asarray(predictions, dtype=np.float32)
targets = np.asarray(targets, dtype=np.float32)

pdj = compute_pdj(predictions, targets, threshold=0.5)
auc = pdj_auc(predictions, targets)

print("frames válidos:", pdj.valid_frames)
print("PDJ@0.5:", f"{pdj.global_score * 100:.2f}%")
print("AUC:", f"{auc * 100:.2f}%")
print("por grupo:")
for group_name, score in pdj.per_group.items():
    print(f"  {group_name}: {score * 100:.2f}%")

frames válidos: 40
PDJ@0.5: 88.53%
AUC: 64.14%
por grupo:
  head: 100.00%
  shoulder: 88.33%
  elbow: 82.50%
  wrist: 80.00%
  hip: 90.83%
  knee: 88.75%
  ankle: 82.50%


## Métricas complementares (PCK, OKS, MPJPE-2D, F1)

In [ ]:
from football_orient_pose.evaluation import (
    compute_pck,
    compute_oks,
    compute_mpjpe_2d,
    joint_detection_report,
)

### PCK@0.2

In [ ]:
pck = compute_pck(predictions, targets, threshold=0.2)
print(f"PCK@0.2: {pck.global_score * 100:.2f}%  (frames válidos: {pck.valid_frames})")
print("\nPCK@0.2 por grupo:")
for group, score in pck.per_group.items():
    print(f"  {group:<12}: {score * 100:.2f}%")

### OKS e AP (padrão COCO)

In [ ]:
oks_result = compute_oks(predictions, targets)
print(f"OKS:     {oks_result.global_oks * 100:.2f}%")
print(f"AP50:    {oks_result.ap50 * 100:.2f}%")
print(f"AP75:    {oks_result.ap75 * 100:.2f}%")
print(f"mAP:     {oks_result.ap * 100:.2f}%")
print("\nmAP por limiar:")
for thr, ap in sorted(oks_result.ap_per_threshold.items()):
    print(f"  @{thr:.2f}  {ap * 100:.2f}%")

### MPJPE-2D (erro médio em pixels)

In [ ]:
mpjpe = compute_mpjpe_2d(predictions, targets)
print(f"MPJPE-2D global: {mpjpe.global_mpjpe:.2f} px")
print("\nErro por grupo (px):")
for group, err in mpjpe.per_group.items():
    print(f"  {group:<12}: {err:.2f} px")

### Detection Report — F1 por joint

In [ ]:
det = joint_detection_report(predictions, targets, threshold=0.5)
print(f"F1-macro: {det.f1_macro * 100:.2f}%")
print("\nF1 por grupo anatômico:")
for group, score in det.per_group.items():
    print(f"  {group:<12}: {score * 100:.2f}%")

### Bar chart: PDJ / PCK / F1 por grupo

In [ ]:
import matplotlib.pyplot as plt

groups   = list(pdj.per_group.keys())
pdj_vals = [pdj.per_group[g] * 100 for g in groups]
pck_vals = [pck.per_group[g] * 100 for g in groups]
f1_vals  = [det.per_group[g] * 100 for g in groups]

x, width = np.arange(len(groups)), 0.25
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, pdj_vals, width, label='PDJ@0.5')
ax.bar(x,          pck_vals, width, label='PCK@0.2')
ax.bar(x + width,  f1_vals,  width, label='F1@0.5')
ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=20)
ax.set_ylabel('Score (%)')
ax.set_title('RTMPose — PDJ / PCK / F1 por grupo anatômico')
ax.set_ylim(0, 105)
ax.legend()
fig.tight_layout()
plt.show()

### Heatmap: MPJPE-2D por joint

In [ ]:
from football_orient_pose.utils.keypoint_mapping import H3WB17_NAMES

fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(mpjpe.per_joint[np.newaxis, :], cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(17))
ax.set_xticklabels(H3WB17_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_yticks([])
ax.set_title('MPJPE-2D por joint (px) — RTMPose zero-shot')
plt.colorbar(im, ax=ax, label='px')
fig.tight_layout()
plt.show()

### Tabela resumo — todas as métricas

In [ ]:
print("\n" + "="*55)
print(f"  {'Métrica':<20}  {'Score':>10}  {'Ref (paper)':>12}")
print("="*55)
print(f"  {'PDJ@0.5':<20}  {pdj.global_score*100:>9.2f}%  {'89.51%':>12}")
print(f"  {'PCK@0.2':<20}  {pck.global_score*100:>9.2f}%  {'—':>12}")
print(f"  {'OKS':<20}  {oks_result.global_oks*100:>9.2f}%  {'—':>12}")
print(f"  {'AP50':<20}  {oks_result.ap50*100:>9.2f}%  {'—':>12}")
print(f"  {'mAP@[.5:.95]':<20}  {oks_result.ap*100:>9.2f}%  {'—':>12}")
print(f"  {'MPJPE-2D':<20}  {mpjpe.global_mpjpe:>9.2f}px  {'—':>12}")
print(f"  {'F1-macro':<20}  {det.f1_macro*100:>9.2f}%  {'—':>12}")
print("="*55)

## Critério para fechar a US #10

- `RTMPoseEstimator()` instancia com `device="cuda"`.
- `predict()` retorna `(17, 3)` com confidences positivas em imagem real.
- `predict_h3wb()` retorna `(17, 2)`.
- Run completo retorna PDJ@0.5 dentro de ±2 pontos percentuais de 89.51%, ou o desvio fica documentado aqui.